In [ ]:
# 04_ramp_shock_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# ramp-shock target (same feature pipeline as 03_violation_baseline.ipynb)
# !pip install lightgbm -q

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "ramp_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()

# --- Time-aware split (same as 03_violation_baseline.ipynb) ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1,
    scale_pos_weight=f.scale_pos_weight(y_train),
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"\nPR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

precision, recall, thresh = precision_recall_curve(y_test, proba_test)
idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0) ---
# PR-AUC 0.7140 vs a random/base-rate baseline of 0.1793 -- roughly 4x lift over chance,
# a genuinely strong baseline signal, unlike the violation target. F1@0.5 = 0.6293 (a
# 0.5 threshold is usable here, unlike violation, because the positive rate is a much
# less extreme ~15-18%). Recall at >=95% precision = 0.1771: even in a
# high-confidence-only alert mode, the model still catches ~18% of upcoming ramp-shocks
# with 15-60 min of lead time.
#
# Top features are dominated by time-of-day ("hour") and the slot's own recent demand
# trajectory (demand_delta_mw, demand_met_mw_lag1/lag3) -- consistent with 01_eda.ipynb's
# finding that ramp-shocks cluster sharply around sunrise (05:00-09:00) and sunset
# (17:00-20:00) solar transitions. Corridor columns (ir_er_wr_net_mu, xb_net_nepal_mu,
# xb_net_bangladesh_mu, ir_er_nr_net_mu) also place in the top 10 -- weaker than the
# time/demand signal but a real, non-trivial contribution, consistent with Era 2's
# daily-resolution finding that corridor flow has a genuine relationship with grid
# stress even though it isn't the dominant driver.
#
# This target is markedly easier to predict with lead time than frequency violation
# (see 03_violation_baseline.ipynb) -- plausibly because a ramp-shock is a direct,
# mechanical property of the demand/generation trajectory itself, while a frequency
# violation is a downstream consequence that depends on how well AGC/reserves absorb a
# given ramp, adding a layer of noise the raw features here don't capture.
